# District Development Profiling from NFHS-5
### Clustering ~700 Indian districts into development profiles

**Runs top to bottom in Google Colab or Jupyter. Total run time ~4 minutes.**
Nothing needs to be downloaded by hand - cell 2 fetches the data.

**The question.** NFHS-5 (2019-21) publishes ~100 health and development
indicators for every district. Policy normally squashes them into one score and
ranks districts. That says *how far behind* a district is, never *what is wrong
with it*. Two districts with the same score can need completely different things.

**What this notebook does.** Groups districts by the *shape* of their
development (clustering), then tests that grouping against the traditional
alternatives - a composite index, and grouping by state - using ten nutrition and
anaemia indicators that no grouping is ever allowed to see.

**Cell map**

| Cells | Phase |
|---|---|
| 1-3 | setup, download, inspect both mirrors |
| 4-8 | cleaning: align the two sources, pivot, impute, de-duplicate |
| 9-11 | EDA: missingness, correlation, within-state spread |
| 12-14 | standardise + PCA |
| 15-16 | traditional baselines |
| 17-20 | clustering: 4 algorithms, choosing k |
| 21-24 | evaluation: held-out outcome test, stability, comparison |
| 25-27 | cluster profiles and names |
| 28 | how safe is each district's label? |
| 29-31 | assigning new districts, unseen-state test, saving everything |

## Cell 1 - Setup

Install anything missing (Colab already has most of it), import, and fix the random seed so every run gives identical numbers.

In [ ]:
import importlib, subprocess, sys

def ensure(pip_name, import_name=None):
    """Install a package only if it is not already importable."""
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)

for pip_name, import_name in [("pandas", None), ("numpy", None), ("scikit-learn", "sklearn"),
                              ("scipy", None), ("matplotlib", None), ("seaborn", None),
                              ("joblib", None), ("tabulate", None)]:
    ensure(pip_name, import_name)

import json, re, urllib.request, warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.impute import KNNImputer
from sklearn.metrics import (adjusted_rand_score, calinski_harabasz_score,
                             davies_bouldin_score, silhouette_score)
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160, "display.max_columns", 40)

SEED = 42                 # every random step uses this -> identical results each run
np.random.seed(SEED)

ROOT = Path("nfhs_output")
for sub in ("raw", "processed", "models", "figures"):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)

print("setup done ->", ROOT.resolve())

## Cell 2 - Download the data

`India.csv` is the primary file: every district, both survey rounds.
`districts.csv` from a second mirror is used only for its **reliability flags** - which estimates came from very few respondents.

In [ ]:
SOURCES = {
    "India.csv":
        "https://raw.githubusercontent.com/SaiSiddhardhaKalla/NFHS/main/India.csv",
    "factsheets.csv":
        "https://raw.githubusercontent.com/jvargh7/nfhs5_factsheets/main/data%20for%20analysis/districts.csv",
}

for name, url in SOURCES.items():
    dest = ROOT / "raw" / name
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] {name} ({dest.stat().st_size/1e6:.1f} MB)")
        continue
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"[ok]   {name} ({dest.stat().st_size/1e6:.1f} MB)")
    except Exception as exc:
        print(f"[fail] {name}: {exc}")

## Cell 3 - What did we get?

Two mirrors of the same NFHS-5 factsheets, and we use each for what it is good at.

**`factsheets.csv` (jvargh7) is the source of VALUES.** Its numbers are already
numeric - no bracketed `(45.2)` to parse, no `*` - and, crucially, it keeps the
reliability information NFHS prints, in its own column: "based on 25-49 unweighted
cases", and "percentage not shown; fewer than 25 unweighted cases".

**`India.csv` (SaiSiddhardhaKalla) supplies Census 2011 district codes and clean
indicator names**, and acts as the reference we align the other file against.

Both are in **long** format: one row per (district, indicator). Machine learning
needs **wide**: one row per district. Cell 6 does that pivot.

In [ ]:
raw = pd.read_csv(ROOT / "raw" / "India.csv", low_memory=False, dtype=str)
for col in ("State", "District", "Indicator", "Category"):
    raw[col] = raw[col].astype(str).str.strip()     # 'Balod ' and 'Balod' are one district

fs_raw = pd.read_csv(ROOT / "raw" / "factsheets.csv", low_memory=False)

print("--- India.csv (reference: census codes + clean indicator names)")
print("  rows x columns :", raw.shape)
print("  states / UTs   :", raw["State"].nunique())
print("  districts      :", raw.groupby(["State", "District"]).ngroups)
print("  indicators     :", raw["Indicator"].nunique())

print("\n--- factsheets.csv (values + reliability flags)")
print("  rows x columns :", fs_raw.shape)
print("  districts      :", fs_raw.groupby(["state", "district"]).ngroups)
print("  columns        :", list(fs_raw.columns))
print("\n  NFHS5 column dtype:", fs_raw["NFHS5"].dtype, "- already numeric, nothing to parse")
print("  reliability flags:")
print(fs_raw["Flag_NFHS5"].value_counts(dropna=False).to_string())
display(fs_raw.head(3))

## Cell 4 - The judgement layer: indicator metadata

This is the most important human decision in the project. Every indicator gets:

* **domain** - which theme it belongs to (education, WASH, nutrition...).
* **direction** - `+1` if higher is better (literacy), `-1` if higher is worse (stunting).
  Needed because you cannot average numbers that point in opposite directions.
* **role**
  * `feature` - used to build the clusters
  * `outcome` - **held out**, never shown to any model, kept to test the clusters later
  * `drop` - not used

**Why these outcomes?** Child stunting, wasting, underweight, overweight and the five
anaemia measures are what policy actually cares about. If clusters built only from
living conditions and services still separate districts on results they never saw,
the clusters are capturing something real.

**Why drop the family-planning method mix?** More condoms is not "better" or "worse"
than more IUDs - no honest direction exists. The totals are kept.

In [ ]:
def canon(name: str) -> str:
    """lower-case, letters and digits only - collapses spelling variants into one key."""
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

# (full_name, short_name, domain, direction, role)
INDICATORS = [
    ("Female population age 6 years and above who ever attended school (%)", "female_ever_attended_school", "education", +1, "feature"),
    ("Population below age 15 years (%)", "pop_below_15", "population_household", -1, "feature"),
    ("Sex ratio of the total population (females per 1,000 males)", "sex_ratio_total", "population_household", +1, "feature"),
    ("Sex ratio at birth for children born in the last five years (females per 1,000 males)", "sex_ratio_at_birth", "population_household", +1, "feature"),
    ("Children under age 5 years whose birth was registered with the civil authority (%)", "birth_registered", "population_household", +1, "feature"),
    ("Deaths in the last 3 years registered with the civil authority (%)", "death_registered", "population_household", +1, "feature"),
    ("Population living in households with electricity (%)", "electricity", "energy", +1, "feature"),
    ("Population living in households with an improved drinkingwater source (%)", "improved_water", "wash", +1, "feature"),
    ("Population living in households that use an improved sanitation facility (%)", "improved_sanitation", "wash", +1, "feature"),
    ("Households using clean fuel for cooking (%)", "clean_cooking_fuel", "energy", +1, "feature"),
    ("Households using iodized salt (%)", "iodized_salt", "nutrition", +1, "feature"),
    ("Households with any usual member covered under a health insurance/financing scheme (%)", "health_insurance", "population_household", +1, "feature"),
    ("Children age 5 years who attended preprimary school during the school year 2019-20 (%)", "preprimary_school", "education", +1, "feature"),
    ("Women who are literate (%)", "women_literate", "education", +1, "feature"),
    ("Women with 10 or more years of schooling (%)", "women_10yr_schooling", "education", +1, "feature"),
    ("Women age 2024 years married before age 18 years (%)", "married_before_18", "women_empowerment", -1, "feature"),
    ("Births in the 5 years preceding the survey that are third or higher order (%)", "birth_order_3plus", "women_empowerment", -1, "feature"),
    ("Women age 15-19 years who were already mothers or pregnant at the time of the survey (%)", "teen_pregnancy", "women_empowerment", -1, "feature"),
    ("Women age 15-24 years who use hygienic methods of protection during their menstrual period (%)", "hygienic_menstrual_protection", "women_empowerment", +1, "feature"),
    ("Any method (%)", "fp_any_method", "women_empowerment", +1, "feature"),
    ("Any modern method (%)", "fp_any_modern_method", "women_empowerment", +1, "feature"),
    ("Female sterilization (%)", "fp_female_sterilization", "women_empowerment", -1, "drop"),
    ("Male sterilization (%)", "fp_male_sterilization", "women_empowerment", +1, "drop"),
    ("IUD/PPIUD (%)", "fp_iud", "women_empowerment", +1, "drop"),
    ("Injectables (%)", "fp_injectables", "women_empowerment", +1, "drop"),
    ("Pill (%)", "fp_pill", "women_empowerment", +1, "drop"),
    ("Condom (%)", "fp_condom", "women_empowerment", +1, "drop"),
    ("Total unmet need (%)", "fp_unmet_need_total", "women_empowerment", -1, "feature"),
    ("Unmet need for spacing (%)", "fp_unmet_need_spacing", "women_empowerment", -1, "feature"),
    ("Current users ever told about side effects of current method (%)", "fp_told_side_effects", "women_empowerment", +1, "feature"),
    ("Health worker ever talked to female non-users about family planning (%)", "fp_worker_talked_to_nonusers", "women_empowerment", +1, "feature"),
    ("Mothers who had an antenatal checkup in the first trimester (%)", "anc_first_trimester", "maternal_health", +1, "feature"),
    ("Mothers who had at least 4 antenatal care visits (%)", "anc_4plus_visits", "maternal_health", +1, "feature"),
    ("Mothers whose last birth was protected against neonatal tetanus (%)", "tetanus_protected_birth", "maternal_health", +1, "feature"),
    ("Mothers who consumed iron folic acid for 100 days or more when they were pregnant (%)", "ifa_100_days", "maternal_health", +1, "feature"),
    ("Mothers who consumed iron folic acid for 180 days or more when they were pregnant (%)", "ifa_180_days", "maternal_health", +1, "feature"),
    ("Registered pregnancies for which the mother received a Mother and Child Protection (MCP) card (%)", "mcp_card_received", "maternal_health", +1, "feature"),
    ("Mothers who received postnatal care from a doctor/nurse/LHV/ANM/midwife/other health personnel within 2 days of delivery (%)", "pnc_mother_2days", "maternal_health", +1, "feature"),
    ("Children who received postnatal care from a doctor/nurse/LHV/ANM/midwife/other health personnel within 2 days of delivery (%)", "pnc_newborn_2days", "maternal_health", +1, "feature"),
    ("Children born at home who were taken to a health facility for a checkup within 24 hours of birth (%)", "home_birth_checkup_24h", "maternal_health", +1, "feature"),
    ("Average out-of-pocket expenditure per delivery in a public health facility (Rs)", "oop_delivery_public_rs", "maternal_health", -1, "feature"),
    ("Institutional births (%)", "institutional_births", "maternal_health", +1, "feature"),
    ("Institutional births in public facility (%)", "institutional_births_public", "maternal_health", +1, "feature"),
    ("Home births that were conducted by skilled health personnel (%)", "home_births_skilled", "maternal_health", +1, "feature"),
    ("Births attended by skilled health personnel (%)", "skilled_birth_attendance", "maternal_health", +1, "feature"),
    ("Births delivered by caesarean section (%)", "caesarean_total", "maternal_health", -1, "feature"),
    ("Births in a private health facility that were delivered by caesarean section (%)", "caesarean_private", "maternal_health", -1, "feature"),
    ("Births in a public health facility that were delivered by caesarean section (%)", "caesarean_public", "maternal_health", -1, "feature"),
    ("Children age 12-23 months fully vaccinated based on information from either vaccination card or mothers recall (%)", "fully_vaccinated_card_or_recall", "child_health_immunisation", +1, "feature"),
    ("Children age 12-23 months fully vaccinated based on information from vaccination card only (%)", "fully_vaccinated_card_only", "child_health_immunisation", +1, "feature"),
    ("Children age 12-23 months who have received 3 doses of penta or DPT vaccine (%)", "dpt3_penta3", "child_health_immunisation", +1, "feature"),
    ("Children age 12-23 months who have received 3 doses of penta or hepatitis B vaccine (%)", "hepb3_penta3", "child_health_immunisation", +1, "feature"),
    ("Children age 12-23 months who have received 3 doses of polio vaccine (%)", "polio3", "child_health_immunisation", +1, "feature"),
    ("Children age 12-23 months who have received 3 doses of rotavirus vaccine (%)", "rotavirus3", "child_health_immunisation", +1, "feature"),
    ("Children age 12-23 months who have received BCG (%)", "bcg", "child_health_immunisation", +1, "feature"),
    ("Children age 12-23 months who have received the first dose of measlescontaining vaccine (MCV) (%)", "measles_dose1", "child_health_immunisation", +1, "feature"),
    ("Children age 24-35 months who have received a second dose of measlescontaining vaccine (MCV) (%)", "measles_dose2", "child_health_immunisation", +1, "feature"),
    ("Children age 12-23 months who received most of their vaccinations in a public health facility (%)", "vaccinated_public_facility", "child_health_immunisation", +1, "feature"),
    ("Children age 12-23 months who received most of their vaccinations in a private health facility (%)", "vaccinated_private_facility", "child_health_immunisation", -1, "feature"),
    ("Children age 9-35 months who received a vitamin A dose in the last 6 months (%)", "vitamin_a_dose", "child_health_immunisation", +1, "feature"),
    ("Children with diarrhoea in the 2 weeks preceding the survey taken to a health facility or health provider (%)", "diarrhoea_care_sought", "child_health_immunisation", +1, "feature"),
    ("Children with diarrhoea in the 2 weeks preceding the survey who received oral rehydration salts (ORS) (%)", "diarrhoea_ors", "child_health_immunisation", +1, "feature"),
    ("Children with diarrhoea in the 2 weeks preceding the survey who received zinc (%)", "diarrhoea_zinc", "child_health_immunisation", +1, "feature"),
    ("Children with fever or symptoms of ARI in the 2 weeks preceding the survey taken to a health facility or health provider (%)", "ari_fever_care_sought", "child_health_immunisation", +1, "feature"),
    ("Prevalence of diarrhoea in the 2 weeks preceding the survey (%)", "diarrhoea_prevalence", "child_health_immunisation", -1, "feature"),
    ("Prevalence of symptoms of acute respiratory infection (ARI) in the 2 weeks preceding the survey (%)", "ari_prevalence", "child_health_immunisation", -1, "feature"),
    ("Children under age 3 years breastfed within one hour of birth (%)", "early_breastfeeding_1h", "nutrition", +1, "feature"),
    ("Children under age 6 months exclusively breastfed (%)", "exclusive_breastfeeding", "nutrition", +1, "feature"),
    ("Children age 6-8 months receiving solid or semisolid food and breastmilk (%)", "complementary_feeding_6_8m", "nutrition", +1, "feature"),
    ("Breastfeeding children age 6-23 months receiving an adequate diet (%)", "adequate_diet_breastfed", "nutrition", +1, "feature"),
    ("Nonbreastfeeding children age 6-23 months receiving an adequate diet (%)", "adequate_diet_nonbreastfed", "nutrition", +1, "feature"),
    ("Total children age 6-23 months receiving an adequate diet (%)", "adequate_diet_total", "nutrition", +1, "feature"),
    ("Children under 5 years who are stunted (height for age) (%)", "child_stunted", "nutrition", -1, "outcome"),
    ("Children under 5 years who are wasted (weight for height) (%)", "child_wasted", "nutrition", -1, "outcome"),
    ("Children under 5 years who are severely wasted (weight for height) (%)", "child_severely_wasted", "nutrition", -1, "outcome"),
    ("Children under 5 years who are underweight (weight for age) (%)", "child_underweight", "nutrition", -1, "outcome"),
    ("Children under 5 years who are overweight (weight fo rheight)20 (%)", "child_overweight", "nutrition", -1, "outcome"),
    ("Women whose Body Mass Index (BMI) is below normal (BMI <18)", "women_bmi_below_normal", "nutrition", -1, "feature"),
    ("Women who are overweight or obese", "women_overweight_obese", "nutrition", -1, "feature"),
    ("Women who have high risk waisttohip ratio", "women_high_waist_hip_ratio", "nutrition", -1, "feature"),
    ("Children age 6-59 months who are anaemic", "children_anaemic", "anaemia", -1, "outcome"),
    ("All women age 15-19 years who are anaemic (%)", "women_15_19_anaemic", "anaemia", -1, "outcome"),
    ("All women age 15-49 years who are anaemic (%)", "women_15_49_anaemic", "anaemia", -1, "outcome"),
    ("Nonpregnant women age 15-49 years who are anaemic", "nonpregnant_women_anaemic", "anaemia", -1, "outcome"),
    ("Pregnant women age 15-49 years who are anaemic", "pregnant_women_anaemic", "anaemia", -1, "outcome"),
    ("Female Blood sugar level  high (141-160 mg/dl) (%)", "women_blood_sugar_high", "ncd", -1, "feature"),
    ("Female Blood sugar level  very high (>160 mg/dl) (%)", "women_blood_sugar_very_high", "ncd", -1, "feature"),
    ("Female Blood sugar level  high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%)", "women_blood_sugar_high_or_medicated", "ncd", -1, "feature"),
    ("Male Blood sugar level  high (141-160 mg/dl) (%)", "men_blood_sugar_high", "ncd", -1, "feature"),
    ("Male Blood sugar level  very high (>160 mg/dl) (%)", "men_blood_sugar_very_high", "ncd", -1, "feature"),
    ("Male Blood sugar level  high or very high (>140 mg/dl) or taking medicine to control blood sugar level (%)", "men_blood_sugar_high_or_medicated", "ncd", -1, "feature"),
    ("Female Mildly elevated blood pressure (Systolic 140-159 mm of Hg and/or Diastolic 90-99 mm of Hg) (%)", "women_bp_mildly_elevated", "ncd", -1, "feature"),
    ("Female Moderately or severely elevated blood pressure (%)", "women_bp_moderate_severe", "ncd", -1, "feature"),
    ("Female Elevated blood pressure or taking medicine to control blood pressure (%)", "women_bp_elevated_or_medicated", "ncd", -1, "feature"),
    ("Male Mildly elevated blood pressure (Systolic 140-159 mm of Hg and/or Diastolic 90-99 mm of Hg) (%)", "men_bp_mildly_elevated", "ncd", -1, "feature"),
    ("Male Moderately or severely elevated blood pressure (%)", "men_bp_moderate_severe", "ncd", -1, "feature"),
    ("Male Elevated blood pressure or taking medicine to control blood pressure (%)", "men_bp_elevated_or_medicated", "ncd", -1, "feature"),
    ("Ever undergone a breast examination for breast cancer (%)", "breast_cancer_exam", "ncd", +1, "feature"),
    ("Ever undergone a screening test for cervical cancer (%)", "cervical_cancer_screening", "ncd", +1, "feature"),
    ("Ever undergone an oral cavity examination for oral cancer (%)", "oral_cancer_exam", "ncd", +1, "feature"),
    ("Men age 15 years and above who use any kind of tobacco (%)", "men_tobacco_use", "tobacco_alcohol", -1, "feature"),
    ("Men age 15 years and above who consume alcohol (%)", "men_alcohol_use", "tobacco_alcohol", -1, "feature"),
    ("Women age 15 years and above who use any kind of tobacco (%)", "women_tobacco_use", "tobacco_alcohol", -1, "feature"),
    ("Women age 15 years and above who consume alcohol (%)", "women_alcohol_use", "tobacco_alcohol", -1, "feature"),
]

DOMAIN_LABELS = {
    "population_household": "Population & household", "education": "Education",
    "wash": "Water & sanitation (WASH)", "energy": "Energy",
    "maternal_health": "Maternal health", "child_health_immunisation": "Child health & immunisation",
    "nutrition": "Nutrition", "anaemia": "Anaemia", "ncd": "NCDs (blood sugar, BP, screening)",
    "women_empowerment": "Women's empowerment & family planning", "tobacco_alcohol": "Tobacco & alcohol",
}

cfg = pd.DataFrame([{"code": canon(f), "short_name": s, "full_name": f, "domain": d,
                     "domain_label": DOMAIN_LABELS[d], "direction": dirn, "role": r}
                    for f, s, d, dirn, r in INDICATORS])

# safety net: every indicator in the data must appear in the table above
missing = {canon(i) for i in raw["Indicator"].unique()} - set(cfg["code"])
assert not missing, f"indicators with no metadata: {missing}"

print(cfg["role"].value_counts().to_string())
print("\ndomains:")
print(cfg["domain"].value_counts().to_string())
display(cfg.head(5))

## Cell 5 - Load the values, and the reliability flags that come with them

NFHS prints an estimate based on only 25-49 respondents in brackets, and refuses
to print one based on fewer than 25 at all. `factsheets.csv` preserves both facts
in its `Flag_NFHS5` column, so we do not have to reconstruct them:

| Flag_NFHS5 | meaning | what we do |
|---|---|---|
| blank | ordinary estimate | keep |
| "Based on 25-49 unweighted cases" | small sample | keep, but flag it |
| "Percentage not shown; ... fewer than 25" | suppressed by NFHS | already blank -> missing |

**Indicators are keyed by their factsheet ITEM NUMBER, not by name.** The names in
this file come straight out of the PDFs, and two things go wrong with them: the
gender of the NCD indicators lives only in the item number (86-88 are women's
blood sugar, 89-91 men's - the names are identical), and a few names are garbled
by the text extraction. Item numbers are reliable; names are not.

In [ ]:
fs = pd.read_csv(ROOT / "raw" / "factsheets.csv", low_memory=False)

# item number: normally leads the string ('86. Blood sugar level - high...').
# In two rows the PDF glued a section heading in front, so fall back to the LAST
# 'NN.' in the text, which is the item's own number.
item = fs["Indicator"].str.extract(r"^\s*(\d+)\s*\.")[0]
fallback = fs["Indicator"].str.findall(r"(\d+)\s*\.\s*[A-Za-z]").str[-1]
fs["item"] = item.fillna(fallback).astype(float)
assert fs["item"].notna().all()

flag_text = fs["Flag_NFHS5"].astype(str)
fs["low_reliability"] = flag_text.str.contains("25-49", na=False).astype(int)
fs["suppressed"] = flag_text.str.contains("not shown", na=False)

def canon(name: str) -> str:
    """lower-case, letters and digits only - collapses spelling variants."""
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

# join key that survives spelling differences: 'kerala|ernakulam'
import difflib
reference_states = sorted({canon(x) for x in raw["State"].unique()})
state_map = {x: (difflib.get_close_matches(canon(x), reference_states, n=1, cutoff=0.6)
                 or [canon(x)])[0] for x in fs["state"].unique()}
fs["key"] = fs["state"].map(state_map) + "|" + fs["district"].map(canon)

values5 = fs.pivot_table(index="key", columns="item", values="NFHS5", aggfunc="mean")
values4 = fs.pivot_table(index="key", columns="item", values="NFHS4", aggfunc="mean")
flags = fs.pivot_table(index="key", columns="item", values="low_reliability", aggfunc="max")
names_tbl = fs.drop_duplicates("key")[["key", "state", "district"]].set_index("key").sort_index()

print(f"values: {values5.shape[0]} districts x {values5.shape[1]} factsheet items")
print(f"low-reliability cells (25-49 cases) : {int(fs['low_reliability'].sum()):,}")
print(f"suppressed cells (already blank)    : {int(fs['suppressed'].sum()):,}")
print(f"missing with no suppression flag    : {int((fs['NFHS5'].isna() & ~fs['suppressed']).sum()):,}")

# --- the reference file: value fingerprints + Census 2011 codes --------------
raw["value"] = pd.to_numeric(raw["NFHS 5"], errors="coerce")
raw = raw[~raw["District"].str.lower().str.contains("not available|unknown", regex=True)]
raw["key"] = raw["State"].map(canon) + "|" + raw["District"].map(canon)
reference = raw.pivot_table(index="key", columns="Indicator", values="value", aggfunc="mean")

st_code = pd.to_numeric(raw["ST_CEN_CD"], errors="coerce")
dt_code = pd.to_numeric(raw["DT_CEN_CD"], errors="coerce")
codes = (raw.assign(census_code=st_code * 1000 + dt_code).groupby("key")["census_code"]
            .agg(lambda x: x.dropna().iloc[0] if x.notna().any() else np.nan).to_frame())
print(f"\nreference: {reference.shape[0]} districts x {reference.shape[1]} named indicators")

## Cell 6 - Aligning the two mirrors by VALUE, not by name

We need to know which of our 104 indicators each factsheet item number is. Name
matching cannot do it (the names are garbled and drop the gender), so we match on
the numbers themselves:

> for each factsheet item, find the India.csv indicator whose values are closest
> across every district the two files share.

Both files were parsed from the same PDFs, so the true match agrees to about
0.001 while the runner-up is off by a thousand times more. That gap is what lets
the code **assert** the alignment instead of trusting it - and it is how we
discover, from the data, that items 86-88 are the women's blood sugar rows and
89-91 the men's.

In [ ]:
name_to_short = {canon(f): short for f, short, *_ in INDICATORS}
shared = values5.index.intersection(reference.index)
prim, ref = values5.loc[shared], reference.loc[shared]
print(f"aligning over {len(shared)} districts present in both files")

mapping, report = {}, []
for it in prim.columns:
    diffs = ref.sub(prim[it], axis=0).abs().mean().dropna().sort_values()
    if len(diffs) < 2:
        continue
    best, runner_up = diffs.index[0], diffs.iloc[1]
    short = name_to_short.get(canon(best))
    if short is None:
        continue
    # unambiguous = the winner is at least 10x closer than the runner-up
    assert diffs.iloc[0] * 10 < runner_up, f"item {int(it)} is ambiguous"
    mapping[it] = short
    report.append(dict(item=int(it), short_name=short,
                       mean_abs_diff=diffs.iloc[0], runner_up_diff=runner_up))

alignment = pd.DataFrame(report).sort_values("item")
print(f"matched {len(mapping)} items to {alignment['short_name'].nunique()} indicators")
print(f"worst alignment error: {alignment['mean_abs_diff'].max():.4f}")
display(alignment[alignment["item"].between(86, 91)])   # the gender proof

def rename(mat):
    out = mat[[c for c in mat.columns if c in mapping]].rename(columns=mapping)
    return out.T.groupby(level=0).mean().T            # collapse repeated items

wide5, wide4, wide_flags = rename(values5), rename(values4), rename(flags)

ids = names_tbl.join(codes, how="left").reset_index()
ids["census_code"] = ids["census_code"].astype("Int64")
# reindex, not .loc: districts created after NFHS-4 have no 2015-16 row at all
wide5 = wide5.reindex(ids["key"]).reset_index(drop=True)
wide4 = wide4.reindex(ids["key"]).reset_index(drop=True)
ids = ids.drop(columns="key")

print(f"\nwide table: {wide5.shape[0]} districts x {wide5.shape[1]} indicators")
print(f"with a Census 2011 code: {ids['census_code'].notna().sum()}")
display(wide5.iloc[:3, :6])

## Cell 7 - Missing data: drop, then impute

Two rules, applied in this order:

1. an indicator missing in **more than 10%** of districts is dropped - imputing most
   of a column would mean inventing it;
2. a district missing **more than 20%** of indicators is dropped.

Watch what this drops. Now that every indicator is flagged consistently,
**anaemia in pregnant women** turns out to be suppressed in 19% of districts -
pregnant women are a small subgroup in a district sample - so it is dropped from
the held-out outcomes, leaving 9. Under name-based flag matching it would have
slipped through with small-sample values intact.

Everything left is filled with **KNN imputation (k=5)**: a gap is filled with the
average of the 5 most similar districts. We standardise before measuring
similarity, otherwise an indicator in rupees (thousands) would drown out
percentages (0-100).

**Features and outcomes are imputed separately.** If outcome columns helped fill
feature columns, the held-out test later would be contaminated.

In [ ]:
feature_cols = [c for c in cfg.loc[cfg["role"] == "feature", "short_name"] if c in wide5.columns]
outcome_cols = [c for c in cfg.loc[cfg["role"] == "outcome", "short_name"] if c in wide5.columns]
X_raw, Y_raw = wide5[feature_cols].copy(), wide5[outcome_cols].copy()

def drop_sparse(mat, ids_, max_col=0.10, max_row=0.20, label=""):
    miss_col = mat.isna().mean().sort_values(ascending=False)
    drop_cols = miss_col[miss_col > max_col].index.tolist()
    print(f"[{label}] overall missing: {mat.isna().to_numpy().mean():.2%}")
    if drop_cols:
        print(f"[{label}] dropping {len(drop_cols)} indicator(s):")
        print(miss_col.head(len(drop_cols)).map(lambda v: f"{v:.1%}").to_string())
    mat = mat.drop(columns=drop_cols)
    miss_row = mat.isna().mean(axis=1)
    drop_rows = miss_row[miss_row > max_row].index
    if len(drop_rows):
        print(f"[{label}] dropping districts:",
              ids_.loc[drop_rows].apply(lambda r: f"{r['state']}/{r['district']}", axis=1).tolist())
    return mat.drop(index=drop_rows), drop_rows

X_raw, dropped_rows = drop_sparse(X_raw, ids, label="features")
Y_raw = Y_raw.drop(index=dropped_rows)
ids = ids.drop(index=dropped_rows)
Y_raw, _ = drop_sparse(Y_raw, ids, label="outcomes")

def knn_impute(mat, label):
    """scale -> KNN fill -> scale back, so the saved table is in real units."""
    pipe = Pipeline([("scale", StandardScaler()), ("impute", KNNImputer(n_neighbors=5))])
    z = pipe.fit_transform(mat)
    filled = pipe.named_steps["scale"].inverse_transform(z)
    print(f"[{label}] imputed {int(mat.isna().sum().sum())} cells")
    return pd.DataFrame(filled, index=mat.index, columns=mat.columns), pipe

X_missing_mask = X_raw.isna()          # kept for the EDA figure in cell 9
X, impute_pipe = knn_impute(X_raw, "features")
Y, _ = knn_impute(Y_raw, "outcomes")
ids = ids.reset_index(drop=True); X = X.reset_index(drop=True); Y = Y.reset_index(drop=True)
print("\nafter cleaning:", X.shape[0], "districts,", X.shape[1], "features,", Y.shape[1], "held-out outcomes")

## Cell 8 - Winsorising and near-duplicate removal

**Winsorising.** A few tiny districts (Chandigarh, Lakshadweep) sit 15+ standard
deviations out on single indicators. Left alone, one district pulls an entire
cluster onto itself. Clipping at the 1st/99th percentile keeps every district in
the analysis while stopping any one value from dominating.

**Near-duplicates.** Two columns correlated above 0.95 are measuring the same
thing twice, which doubles that concept's weight in every distance and in PCA.
We keep whichever is listed earlier in the metadata table - the headline version
rather than the sub-component.

In [ ]:
lo, hi = X.quantile(0.01), X.quantile(0.99)
n_clipped = int(((X < lo) | (X > hi)).to_numpy().sum())
X = X.clip(lower=lo, upper=hi, axis=1)
print(f"winsorised {n_clipped} cells ({n_clipped / X.size:.1%})")

order = {s: i for i, s in enumerate(cfg["short_name"])}
corr_abs = X.corr().abs()
dropped = {}
for i, a in enumerate(corr_abs.columns):
    if a in dropped:
        continue
    for b in corr_abs.columns[i + 1:]:
        if b in dropped:
            continue
        if corr_abs.loc[a, b] > 0.95:
            loser = b if order[a] <= order[b] else a
            keeper = a if loser == b else b
            dropped[loser] = (keeper, corr_abs.loc[a, b])
            if loser == a:
                break
for loser, (keeper, r) in dropped.items():
    print(f"dropped {loser:32s} (r = {r:.3f} with {keeper})")
X = X.drop(columns=list(dropped))
print("\nfinal feature matrix:", X.shape)

## Cell 9 - EDA 1: where were the gaps?

Red = missing before imputation. The gaps are concentrated in **small-denominator** indicators (only children who had diarrhoea in the last two weeks, only non-breastfed infants), which is exactly why NFHS suppressed them.

In [ ]:
plt.figure(figsize=(14, 7))
sns.heatmap(X_missing_mask.T, cbar=False, cmap=["#f0f0f0", "#c0392b"])
plt.title("Missing values before imputation (red = missing)")
plt.xlabel("districts"); plt.xticks([]); plt.yticks(fontsize=5)
plt.tight_layout(); plt.savefig(ROOT / "figures" / "01_missing.png", dpi=120); plt.show()

## Cell 10 - EDA 2: the indicators overlap a lot (why we need PCA)

Columns are ordered by domain. The dark square blocks are groups of indicators that all measure the same underlying thing. That redundancy is the argument for PCA.

In [ ]:
order_by_domain = (cfg[cfg["short_name"].isin(X.columns)]
                   .sort_values(["domain", "short_name"])["short_name"].tolist())
corr = X[order_by_domain].corr()

plt.figure(figsize=(13, 11))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
            xticklabels=True, yticklabels=True, cbar_kws={"shrink": 0.6})
plt.xticks(fontsize=4, rotation=90); plt.yticks(fontsize=4)
plt.title("Feature correlation, grouped by domain")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "02_correlation.png", dpi=120); plt.show()

abs_corr = corr.abs().where(~np.eye(len(corr), dtype=bool))
print(f"mean |correlation|          : {abs_corr.stack().mean():.2f}")
print(f"feature pairs with |r| > 0.6: {int((abs_corr > 0.6).sum().sum() / 2)}")

## Cell 11 - EDA 3: districts inside one state are not alike

If states were internally uniform, 'group by state' would be enough. The share of variance sitting *within* states says otherwise - and that is the reason this project clusters districts rather than states.

In [ ]:
df_states = pd.concat([ids[["state"]], X], axis=1)
rows = []
for col in X.columns:
    grand = X[col].mean()
    between = df_states.groupby("state")[col].apply(lambda s: len(s) * (s.mean() - grand) ** 2).sum()
    total = ((X[col] - grand) ** 2).sum()
    rows.append({"indicator": col, "within_state_share": 1 - between / total})
within = pd.DataFrame(rows).sort_values("within_state_share", ascending=False)
print(f"median within-state share of variance: {within['within_state_share'].median():.2f}")

big = df_states["state"].value_counts()
big = big[big >= 10].index[:16]
plt.figure(figsize=(14, 6))
sub = df_states[df_states["state"].isin(big)]
order_s = sub.groupby("state")["improved_sanitation"].median().sort_values().index
sns.boxplot(data=sub, x="state", y="improved_sanitation", order=order_s, color="#7fcdbb")
plt.xticks(rotation=90, fontsize=8); plt.xlabel("")
plt.title("Sanitation coverage: spread of districts within each state")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "03_within_state.png", dpi=120); plt.show()
display(within.head(5))

## Cell 12 - Standardise, then PCA

**Standardise** (`StandardScaler`): each feature becomes mean 0, standard deviation 1.
Without this, an indicator measured in rupees would dominate every distance.

**PCA**: rotates the 77 correlated features onto new uncorrelated axes, ordered by
how much variation each explains. Component 1 is the single direction along which
districts differ most.

In [ ]:
scaler = StandardScaler().fit(X)
Z = scaler.transform(X)                      # 705 x 77, standardised
pca = PCA(random_state=SEED).fit(Z)
cum = np.cumsum(pca.explained_variance_ratio_)

ks = {t: int(np.searchsorted(cum, t) + 1) for t in (0.80, 0.90, 0.95)}
print(f"PC1 alone explains : {pca.explained_variance_ratio_[0]:.1%}")
print(f"PC1-PC5 explain    : {cum[4]:.1%}")
for t, kc in ks.items():
    print(f"components for {t:.0%} of the variance: {kc}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(range(1, 31), pca.explained_variance_ratio_[:30] * 100, color="#2c7fb8")
axes[0].set_title("Scree plot"); axes[0].set_xlabel("component"); axes[0].set_ylabel("% variance")
axes[1].plot(range(1, len(cum) + 1), cum * 100, marker="o", ms=3)
for t, colour in zip((0.80, 0.90, 0.95), ("#d95f0e", "#c7254e", "#31a354")):
    axes[1].axhline(t * 100, ls="--", color=colour, label=f"{t:.0%} -> {ks[t]} components")
axes[1].set_xlim(0, 60); axes[1].legend(); axes[1].set_xlabel("components")
axes[1].set_title("Cumulative variance explained")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "04_pca_variance.png", dpi=120); plt.show()

P80 = pca.transform(Z)[:, :ks[0.80]]
P90 = pca.transform(Z)[:, :ks[0.90]]
P95 = pca.transform(Z)[:, :ks[0.95]]

## Cell 13 - What are the components actually measuring?

A component is a weighted recipe of the original indicators. Reading the largest weights tells you what the axis means in words.

In [ ]:
for i in range(4):
    s = pd.Series(pca.components_[i], index=X.columns)
    print(f"--- PC{i+1}  ({pca.explained_variance_ratio_[i]:.1%} of all variation)")
    print("   high end:", ", ".join(s.sort_values(ascending=False).head(5).index))
    print("   low end :", ", ".join(s.sort_values().head(5).index))

loadings = pd.DataFrame(pca.components_[:5].T, index=X.columns,
                        columns=[f"PC{i}" for i in range(1, 6)])
plt.figure(figsize=(7, 14))
sns.heatmap(loadings.loc[order_by_domain], cmap="RdBu_r", center=0, yticklabels=True)
plt.yticks(fontsize=5); plt.title("PCA loadings, first 5 components")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "05_loadings.png", dpi=120); plt.show()

## Cell 14 - The districts, drawn in 2 dimensions

PC1 vs PC2 with the biggest states coloured. Notice the states **overlap** - being in the same state does not mean having the same development profile.

In [ ]:
scatter = ids.copy()
scatter["PC1"], scatter["PC2"] = pca.transform(Z)[:, 0], pca.transform(Z)[:, 1]
top_states = scatter["state"].value_counts().head(10).index

plt.figure(figsize=(11, 8))
sns.scatterplot(data=scatter[~scatter["state"].isin(top_states)], x="PC1", y="PC2",
                color="#dddddd", s=18, label="other states")
sns.scatterplot(data=scatter[scatter["state"].isin(top_states)], x="PC1", y="PC2",
                hue="state", palette="tab10", s=35)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.title("Districts in PCA space, coloured by state")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "06_pca_scatter.png", dpi=120); plt.show()

## Cell 15 - Baseline 1 and 2: the composite indices

This is how district ranking is normally done. Flip the indicators where higher
is worse, squeeze every column to 0-1, average them, and cut the result into
equal-sized groups.

The domain-weighted version averages **within each domain first**, so a domain
with 15 indicators does not outvote a domain with 2.

In [ ]:
direction = dict(zip(cfg["short_name"], cfg["direction"]))
domain_of = dict(zip(cfg["short_name"], cfg["domain"]))

good = X.copy()
for c in good.columns:
    if direction[c] < 0:
        good[c] = -good[c]                     # now bigger = better for every column
good = (good - good.min()) / (good.max() - good.min())    # min-max to 0-1

composite = good.mean(axis=1)
by_domain = good.T.groupby(good.columns.map(domain_of)).mean().T
domain_score = by_domain.mean(axis=1)

print(f"Spearman correlation between the two indices: "
      f"{composite.corr(domain_score, method='spearman'):.3f}")
print("\ntop 5 districts by composite score:")
display(ids.assign(score=composite).nlargest(5, "score"))
print("bottom 5:")
display(ids.assign(score=composite).nsmallest(5, "score"))

## Cell 16 - Baseline 3 and 4: state and region

Grouping districts by the state they are in, and by one of six regions. These are how money is usually allocated.

In [ ]:
REGIONS = {
    "North": ["Jammu & Kashmir", "Ladakh", "Himachal Pradesh", "Punjab", "Haryana",
              "Chandigarh", "NCT of Delhi", "Rajasthan", "Uttarakhand"],
    "Central": ["Uttar Pradesh", "Madhya Pradesh", "Chhattisgarh"],
    "East": ["Bihar", "Jharkhand", "Odisha", "West Bengal"],
    "West": ["Gujarat", "Maharashtra", "Goa", "Dadra & Nagar Haveli", "Daman & Diu"],
    "South": ["Andhra Pradesh", "Telangana", "Karnataka", "Kerala", "Tamil Nadu",
              "Puducherry", "Lakshadweep", "Andaman & Nicobar Island"],
    "North-East": ["Assam", "Arunachal Pradesh", "Manipur", "Meghalaya", "Mizoram",
                   "Nagaland", "Tripura", "Sikkim"],
}
# Match on canonical spelling, then fall back to the closest spelling: the
# factsheet mirror writes 'NCT Delhi' and 'Jammu Kashmir' where India.csv writes
# 'NCT of Delhi' and 'Jammu & Kashmir', and it uses the merged UT
# 'Dadra Nagar Haveli Daman Diu'. All of them must land in a region.
canon_to_region = {canon(s): r for r, states in REGIONS.items() for s in states}

def region_of(state):
    key = canon(state)
    if key in canon_to_region:
        return canon_to_region[key]
    match = difflib.get_close_matches(key, list(canon_to_region), n=1, cutoff=0.5)
    return canon_to_region[match[0]] if match else None

base = ids.copy()
base["composite_score"] = composite
base["state_group"] = base["state"].astype("category").cat.codes
base["region"] = base["state"].map(region_of)
assert base["region"].notna().all(),     f"no region for: {sorted(base.loc[base.region.isna(), 'state'].unique())}"
resolved = [f"{s} -> {region_of(s)}" for s in sorted(base["state"].unique())
            if canon(s) not in canon_to_region]
if resolved:
    print("matched by closest spelling:", "; ".join(resolved))
base["region_group"] = base["region"].astype("category").cat.codes
print(base["region"].value_counts().to_string())

## Cell 17 - The four clustering algorithms

* **K-means** - pick k centres, put each district with the nearest centre, move each
  centre to the average of its members, repeat. `n_init=50` because the answer
  depends on where the centres start.
* **Ward (hierarchical)** - start with every district alone and repeatedly merge the
  pair that increases within-group spread least. Gives a tree (dendrogram).
* **Gaussian Mixture** - assumes the data is a blend of stretched Gaussian blobs and
  gives each district a *probability* of belonging to each.
* **DBSCAN** - grows clusters out of dense regions and calls sparse points noise.
  Included on purpose as a contrast.

In [ ]:
def fit_kmeans(Xfit, k):
    return KMeans(n_clusters=k, n_init=50, random_state=SEED).fit(Xfit)

def fit_ward(Xfit, k):
    return AgglomerativeClustering(n_clusters=k, linkage="ward").fit(Xfit)

def fit_gmm(Xfit, k, cov):
    return GaussianMixture(n_components=k, covariance_type=cov, n_init=10,
                           random_state=SEED).fit(Xfit)

def score_all(Zeval, labels):
    """Internal quality, always measured in the same 77-feature space."""
    mask = labels >= 0                                   # ignore DBSCAN noise
    if len(np.unique(labels[mask])) < 2:
        return dict(silhouette=np.nan, davies_bouldin=np.nan, calinski_harabasz=np.nan)
    return dict(silhouette=silhouette_score(Zeval[mask], labels[mask]),
                davies_bouldin=davies_bouldin_score(Zeval[mask], labels[mask]),
                calinski_harabasz=calinski_harabasz_score(Zeval[mask], labels[mask]))

K_RANGE = range(3, 11)
sweep = []
for space, Xfit in (("raw", Z), ("pca", P90)):
    for k in K_RANGE:
        sweep.append(dict(space=space, algo="kmeans", k=k, **score_all(Z, fit_kmeans(Xfit, k).labels_)))
        sweep.append(dict(space=space, algo="ward", k=k, **score_all(Z, fit_ward(Xfit, k).labels_)))
        cov = "full" if space == "pca" else "diag"
        gm = fit_gmm(Xfit, k, cov)
        sweep.append(dict(space=space, algo=f"gmm_{cov}", k=k,
                          **score_all(Z, gm.predict(Xfit)), bic=gm.bic(Xfit)))
sweep = pd.DataFrame(sweep)
display(sweep[sweep.space == "pca"].pivot_table(index="k", columns="algo", values="silhouette").round(3))

## Cell 18 - The usual ways of choosing k, and why they are not enough here

Elbow, silhouette, Davies-Bouldin, Calinski-Harabasz and BIC. Look at the silhouette panel: it is **flat** - every k scores about 0.08-0.11. That is what a continuous spread of districts looks like, and it means we should not read a winner off differences of 0.01.

In [ ]:
inertia = {k: fit_kmeans(P90, k).inertia_ for k in K_RANGE}

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
axes[0, 0].plot(list(inertia), list(inertia.values()), marker="o")
axes[0, 0].set_title("Elbow: K-means inertia"); axes[0, 0].set_xlabel("k")
for ax, metric, title in [(axes[0, 1], "silhouette", "Silhouette (higher better)"),
                          (axes[1, 0], "davies_bouldin", "Davies-Bouldin (lower better)"),
                          (axes[1, 1], "calinski_harabasz", "Calinski-Harabasz (higher better)")]:
    for (space, algo), g in sweep.groupby(["space", "algo"]):
        ax.plot(g["k"], g[metric], marker="o", ms=4, label=f"{algo}/{space}")
    ax.set_title(title); ax.set_xlabel("k"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(ROOT / "figures" / "07_choosing_k.png", dpi=120); plt.show()

plt.figure(figsize=(14, 5))
dendrogram(linkage(P90, method="ward"), truncate_mode="lastp", p=40,
           leaf_rotation=90, leaf_font_size=8)
plt.title("Ward dendrogram (last 40 merges)"); plt.ylabel("merge distance")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "08_dendrogram.png", dpi=120); plt.show()

## Cell 19 - Choosing k by reproducibility instead

Since the silhouette curve is flat, we choose k on a question that actually
matters: **would we get the same groups from a slightly different sample?**

For each k, cluster the full data, then re-cluster 30 random 80% subsamples and
measure the **Adjusted Rand Index** (1 = identical grouping, 0 = chance).

Rule, fixed in advance and using **no outcome data**:
every cluster must have at least 15 districts, mean ARI must stay above 0.60, and
we take the largest k before stability first breaks.

In [ ]:
MIN_CLUSTER_SIZE, MIN_STABILITY_ARI = 15, 0.60
rng = np.random.default_rng(SEED)
stab_rows = []
for k in K_RANGE:
    basel = fit_kmeans(P90, k).labels_
    aris = []
    for b in range(30):
        idx = rng.choice(len(P90), int(0.8 * len(P90)), replace=False)
        sub_labels = KMeans(n_clusters=k, n_init=20, random_state=b).fit(P90[idx]).labels_
        aris.append(adjusted_rand_score(basel[idx], sub_labels))
    stab_rows.append(dict(k=k, stability_ari=float(np.mean(aris)), stability_sd=float(np.std(aris))))
stab = pd.DataFrame(stab_rows)

size_ok = [k for k in K_RANGE
           if min(pd.Series(fit_kmeans(P90, k).labels_).value_counts().min(),
                  pd.Series(fit_ward(P90, k).labels_).value_counts().min()) >= MIN_CLUSTER_SIZE]
passing = sorted(stab[(stab.k.isin(size_ok)) & (stab.stability_ari >= MIN_STABILITY_ARI)]["k"])
K = passing[0]
for cand in passing[1:]:
    if cand == K + 1:
        K = cand
    else:
        break
print(f"k with all clusters >= {MIN_CLUSTER_SIZE}: {size_ok}")
print(f"k passing the stability floor      : {passing}")
print(f"CHOSEN k = {K}")

plt.figure(figsize=(9, 4.5))
plt.errorbar(stab["k"], stab["stability_ari"], yerr=stab["stability_sd"], marker="o", capsize=4)
plt.axhline(MIN_STABILITY_ARI, ls="--", color="#c7254e", label="stability floor 0.60")
plt.axvline(K, ls=":", color="#31a354", label=f"chosen k = {K}")
plt.xlabel("k"); plt.ylabel("bootstrap ARI"); plt.legend()
plt.title("How reproducible is each k?")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "09_stability_by_k.png", dpi=120); plt.show()

## Cell 20 - Fit every algorithm at the chosen k (plus DBSCAN)

DBSCAN's `eps` comes from the 'knee' of the k-distance curve - the point where neighbour distances start growing quickly.

In [ ]:
labels_all = {}
for space, Xfit in (("raw", Z), ("pca", P90)):
    labels_all[f"kmeans_{space}"] = fit_kmeans(Xfit, K).labels_
    labels_all[f"ward_{space}"] = fit_ward(Xfit, K).labels_
    cov = "full" if space == "pca" else "diag"
    labels_all[f"gmm_{space}"] = fit_gmm(Xfit, K, cov).predict(Xfit)

# --- DBSCAN: choose eps from the k-distance knee ---------------------------
nn = NearestNeighbors(n_neighbors=5).fit(P90)
dists = np.sort(nn.kneighbors(P90)[0][:, -1])
line = dists[0] + (dists[-1] - dists[0]) * np.arange(len(dists)) / (len(dists) - 1)
eps = float(dists[int(np.argmax(line - dists))])

grid = [dict(eps=round(eps * m, 3), min_samples=ms,
             n_clusters=len({l for l in DBSCAN(eps=eps * m, min_samples=ms).fit_predict(P90) if l >= 0}),
             noise=float((DBSCAN(eps=eps * m, min_samples=ms).fit_predict(P90) == -1).mean()))
        for m in (0.75, 1.0, 1.25, 1.5, 2.0) for ms in (5, 10)]
grid = pd.DataFrame(grid)
usable = grid[(grid.n_clusters >= 2) & (grid.noise < 0.5)].sort_values("noise")
pick = usable.iloc[0] if len(usable) else grid.sort_values("noise").iloc[0]
labels_all["dbscan_pca"] = DBSCAN(eps=float(pick.eps), min_samples=int(pick.min_samples)).fit_predict(P90)
display(grid)
print(f"DBSCAN chosen: eps={pick.eps}, min_samples={int(pick.min_samples)} -> "
      f"{int(pick.n_clusters)} clusters, {pick.noise:.1%} noise")

print("\ncluster sizes at k =", K)
for name, lab in labels_all.items():
    print(f"  {name:12s}", pd.Series(lab).value_counts().sort_index().to_dict())

## Cell 21 - The held-out outcome test (the heart of the project)

Ten indicators - stunting, wasting, severe wasting, underweight, overweight and five
anaemia measures - were separated in cell 7 and **never shown to any grouping**.

`eta-squared` is the share of an outcome's variance explained by a grouping:

$$\eta^2 = \frac{\text{variation between groups}}{\text{total variation}}$$

A grouping that captures real differences between districts should separate
outcomes it has never seen. One that just reshuffles districts will not.

In [ ]:
def eta_squared(values, labels):
    """One-way ANOVA effect size: between-group variance / total variance."""
    mask = labels >= 0
    v, l = values[mask], labels[mask]
    grand = v.mean()
    ss_total = ((v - grand) ** 2).sum()
    if ss_total == 0:
        return np.nan
    ss_between = sum(((v[l == g].mean() - grand) ** 2) * (l == g).sum() for g in np.unique(l))
    return float(ss_between / ss_total)

def distinct_features(Xdf, labels):
    """How many features differ significantly between groups (Kruskal-Wallis)."""
    mask = labels >= 0
    groups = np.unique(labels[mask])
    if len(groups) < 2:
        return 0
    alpha = 0.01 / Xdf.shape[1]                  # Bonferroni: 77 tests
    hits = 0
    for col in Xdf.columns:
        samples = [Xdf.loc[mask, col].to_numpy()[labels[mask] == g] for g in groups]
        samples = [s for s in samples if len(s) > 1]
        try:
            if len(samples) >= 2 and stats.kruskal(*samples).pvalue < alpha:
                hits += 1
        except ValueError:
            pass
    return hits

Yz = (Y - Y.mean()) / Y.std()                    # z-scored so outcomes are comparable
quantile_groups = lambda s, k: pd.qcut(s.rank(method="first"), q=k, labels=False).to_numpy()

groupings = dict(labels_all)
groupings["composite_index"] = quantile_groups(composite, K)
groupings["domain_index"] = quantile_groups(domain_score, K)
groupings["state"] = base["state_group"].to_numpy()
groupings["region"] = base["region_group"].to_numpy()
groupings["kmeans_pca80"] = fit_kmeans(P80, K).labels_
groupings["kmeans_pca95"] = fit_kmeans(P95, K).labels_
groupings["kmeans_pca_k3"] = fit_kmeans(P90, 3).labels_
print("groupings to compare:", len(groupings))

## Cell 22 - Stability for every grouping

50 bootstrap samples of 80% of districts. Clusterings are refitted; the index baselines are rebuilt the same way; state and region are deterministic, so their 1.00 is by construction and labelled as such.

In [ ]:
SPACES = {"raw": Z, "pca90": P90, "pca80": P80, "pca95": P95}
SPECS = {}
for name in labels_all:
    algo, space = name.rsplit("_", 1)
    SPECS[name] = dict(algo=algo, space="raw" if space == "raw" else "pca90", k=K)
SPECS["dbscan_pca"].update(eps=float(pick.eps), min_samples=int(pick.min_samples))
SPECS["kmeans_pca80"] = dict(algo="kmeans", space="pca80", k=K)
SPECS["kmeans_pca95"] = dict(algo="kmeans", space="pca95", k=K)
SPECS["kmeans_pca_k3"] = dict(algo="kmeans", space="pca90", k=3)

def fit_one(spec, Xfit):
    if spec["algo"] == "kmeans":
        return KMeans(n_clusters=spec["k"], n_init=20, random_state=SEED).fit(Xfit).labels_
    if spec["algo"] == "ward":
        return AgglomerativeClustering(n_clusters=spec["k"], linkage="ward").fit(Xfit).labels_
    if spec["algo"] == "gmm":
        cov = "full" if spec["space"] != "raw" else "diag"
        return GaussianMixture(n_components=spec["k"], covariance_type=cov, n_init=5,
                               random_state=SEED).fit(Xfit).predict(Xfit)
    return DBSCAN(eps=spec["eps"], min_samples=spec["min_samples"]).fit_predict(Xfit)

def stability(name, labels):
    rng_ = np.random.default_rng(SEED)
    out = []
    for _ in range(50):
        idx = rng_.choice(len(labels), int(0.8 * len(labels)), replace=False)
        if name in ("state", "region"):
            out.append(1.0)
        elif name in ("composite_index", "domain_index"):
            sub = good.iloc[idx]
            rescaled = (sub - sub.min()) / (sub.max() - sub.min())
            out.append(adjusted_rand_score(labels[idx], quantile_groups(rescaled.mean(axis=1), K)))
        else:
            out.append(adjusted_rand_score(labels[idx], fit_one(SPECS[name], SPACES[SPECS[name]["space"]][idx])))
    return float(np.nanmean(out))
print("stability helper ready")

## Cell 23 - The comparison table

This is the table the whole project exists to produce. Takes about a minute.

In [ ]:
rows = []
for name, labels in groupings.items():
    etas = [eta_squared(Yz[c].to_numpy(), labels) for c in Y.columns]
    rows.append(dict(grouping=name,
                     n_groups=int(len(np.unique(labels[labels >= 0]))),
                     smallest_group=int(pd.Series(labels[labels >= 0]).value_counts().min()),
                     **score_all(Z, labels),
                     mean_outcome_eta2=float(np.nanmean(etas)),
                     distinct_features=distinct_features(X, labels),
                     stability_ari=stability(name, labels)))
comparison = pd.DataFrame(rows).sort_values("mean_outcome_eta2", ascending=False)
display(comparison[["grouping", "n_groups", "silhouette", "davies_bouldin",
                    "mean_outcome_eta2", "distinct_features", "stability_ari"]].round(3))

## Cell 24 - Choosing the final model, and reading the result honestly

Selection order, fixed in advance: **a stability floor -> held-out eta-squared ->
stability -> silhouette**, with differences in eta-squared under 0.02 treated as
ties (a gap that small is noise), and any grouping with a cluster under 15
districts excluded.

The stability floor is the same 0.60 used to choose k, and it earns its place:
without it the rule selects a model that scores a fraction higher on eta-squared
while being far less reproducible - and that model then fails the unseen-state
test in cell 30, because both tests ask the same question.

Two things to say out loud when presenting:

* against the like-for-like baseline (the composite index cut into the same number of
  groups) clustering wins by a wide margin;
* **state grouping scores highest** - but with ~34 groups against 8, and eta-squared
  rises mechanically with more groups. States also share diet, policy and health
  systems. Reported, not hidden.

In [ ]:
ETA_TOLERANCE = 0.02
MIN_STABILITY_ARI = 0.60          # the same floor cell 19 used to choose k

candidates = comparison[~comparison.grouping.isin(["composite_index", "domain_index", "state", "region"])]
candidates = candidates[(candidates.smallest_group >= 15) & (candidates.n_groups >= 3)]

# Stability floor. Cell 19 refused to accept a k whose clustering falls apart when
# 20% of districts are removed; the same standard must apply to the algorithm.
# Without it the rule picks whichever model squeezes out the highest eta-squared
# even when it is barely reproducible - and an unreproducible grouping also fails
# the unseen-state test in cell 30, because "refit without one state" is the same
# question as "refit without 20% of districts".
stable = candidates[candidates.stability_ari >= MIN_STABILITY_ARI]
excluded = sorted(set(candidates.grouping) - set(stable.grouping))
print("excluded for stability <", MIN_STABILITY_ARI, ":", excluded)
if stable.empty:
    print("WARNING: nothing cleared the floor; selecting on eta-squared alone")
    stable = candidates

tied = stable[stable.mean_outcome_eta2 >= stable.mean_outcome_eta2.max() - ETA_TOLERANCE]
final_name = tied.sort_values(["stability_ari", "silhouette"], ascending=False).iloc[0]["grouping"]
final_labels = groupings[final_name]
final_space = SPECS[final_name]["space"]
final_scores = SPACES[final_space]
print(f"FINAL MODEL: {final_name}  (k = {K}, space = {final_space})")

show = comparison.set_index("grouping")
print(f"\nheld-out outcome variance explained")
print(f"  this model        : {show.loc[final_name, 'mean_outcome_eta2']:.3f}")
print(f"  composite index   : {show.loc['composite_index', 'mean_outcome_eta2']:.3f}  <- same number of groups")
print(f"  domain index      : {show.loc['domain_index', 'mean_outcome_eta2']:.3f}")
print(f"  state grouping    : {show.loc['state', 'mean_outcome_eta2']:.3f}  <- {int(show.loc['state', 'n_groups'])} groups, not comparable")

plot_rows = comparison[comparison.grouping.isin(
    [final_name, "gmm_raw", "ward_pca", "composite_index", "domain_index", "state", "region", "dbscan_pca"])]
colours = ["#c7254e" if g in ("composite_index", "domain_index", "state", "region") else "#2c7fb8"
           for g in plot_rows.sort_values("mean_outcome_eta2")["grouping"]]
plt.figure(figsize=(9, 5))
d = plot_rows.sort_values("mean_outcome_eta2")
plt.barh(d["grouping"], d["mean_outcome_eta2"], color=colours)
plt.xlabel("held-out outcome variance explained (eta-squared)")
plt.title("Clustering (blue) vs traditional groupings (red)")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "10_comparison.png", dpi=120); plt.show()

## Cell 25 - Same score, different problems

The project's core claim, made visible: districts the composite index calls equal, which the clustering separates because their problems have different shapes.

In [ ]:
z_oriented = (X - X.mean()) / X.std()
for c in z_oriented.columns:
    z_oriented[c] *= direction[c]                # + always means better
district_domains = z_oriented.T.groupby(z_oriented.columns.map(domain_of)).mean().T

cand = ids.assign(score=composite, cluster=final_labels).sort_values("score").reset_index()
pairs = []
for i in range(len(cand) - 1):
    for j in range(i + 1, min(i + 12, len(cand))):
        if cand.loc[j, "score"] - cand.loc[i, "score"] > 0.02:
            break
        if cand.loc[i, "cluster"] == cand.loc[j, "cluster"]:
            continue
        a, b = cand.loc[i, "index"], cand.loc[j, "index"]
        pairs.append(dict(a=a, b=b,
                          name_a=f"{cand.loc[i,'district']} ({cand.loc[i,'state']})",
                          name_b=f"{cand.loc[j,'district']} ({cand.loc[j,'state']})",
                          score_a=cand.loc[i, "score"], score_b=cand.loc[j, "score"],
                          gap=float(np.abs(district_domains.loc[a] - district_domains.loc[b]).max())))
pairs = pd.DataFrame(pairs).sort_values("gap", ascending=False)
print(f"{len(pairs)} district pairs are within 0.02 on the composite index but in different profiles")

top = pairs.head(3)
fig, axes = plt.subplots(len(top), 1, figsize=(12, 4.2 * len(top)))
for ax, (_, r) in zip(np.atleast_1d(axes), top.iterrows()):
    xpos = np.arange(district_domains.shape[1])
    ax.bar(xpos - 0.2, district_domains.loc[r["a"]], 0.4, label=r["name_a"], color="#2c7fb8")
    ax.bar(xpos + 0.2, district_domains.loc[r["b"]], 0.4, label=r["name_b"], color="#d95f0e")
    ax.set_xticks(xpos)
    ax.set_xticklabels([DOMAIN_LABELS[d] for d in district_domains.columns], rotation=25,
                       ha="right", fontsize=8)
    ax.set_title(f"composite {r['score_a']:.3f} vs {r['score_b']:.3f} - different profiles", fontsize=11)
    ax.set_ylabel("domain score (+ = better)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(ROOT / "figures" / "11_same_score.png", dpi=120); plt.show()

## Cell 26 - What is each cluster like?

Average domain score per cluster, with every indicator flipped so **positive always means better than the national average**.

In [ ]:
prof = district_domains.groupby(final_labels).mean()
prof.index.name = "cluster"

plt.figure(figsize=(11, 4.5))
sns.heatmap(prof, annot=True, fmt=".2f", center=0, cmap="RdYlGn", vmin=-1.2, vmax=1.2,
            xticklabels=[DOMAIN_LABELS[d] for d in prof.columns],
            cbar_kws={"label": "mean z-score"})
plt.xticks(rotation=30, ha="right", fontsize=8); plt.ylabel("cluster")
plt.title("Development profiles: mean domain score per cluster")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "12_profiles.png", dpi=120); plt.show()
display(prof.round(2))

## Cell 27 - Naming the clusters and their radar charts

Names are generated **mechanically** from the numbers - a development level plus the
weakest domains - so they follow the data rather than our expectations.

NCDs are left out of the naming logic (not the charts): high blood sugar and blood
pressure *rise* with development, so "high development, weak NCDs" would be nonsense.

In [ ]:
SHORT_DOMAIN = {"population_household": "civil registration", "education": "schooling",
                "wash": "sanitation", "energy": "clean energy", "maternal_health": "maternal care",
                "child_health_immunisation": "child health", "nutrition": "nutrition",
                "anaemia": "anaemia", "ncd": "NCDs", "women_empowerment": "women's agency",
                "tobacco_alcohol": "tobacco & alcohol"}

def name_cluster(profile):
    p = profile.drop(["ncd"], errors="ignore")
    level = p.mean()
    word = "High development" if level > 0.35 else "Low development" if level < -0.35 else "Middle development"
    gaps = p[p < -0.40].sort_values().head(2)
    strengths = p[p > 0.60].sort_values(ascending=False).head(1)
    phrase = {"tobacco_alcohol": "high tobacco & alcohol use", "anaemia": "high anaemia"}
    if len(gaps):
        return f"{word}, " + " and ".join(phrase.get(d, f"weak {SHORT_DOMAIN[d]}") for d in gaps.index)
    if len(strengths):
        return f"{word}, strong " + SHORT_DOMAIN[strengths.index[0]]
    return f"{word}, no standout gap"

names = {int(c): name_cluster(prof.loc[c]) for c in prof.index}
counts = pd.Series(list(names.values())).value_counts()
for dupe in counts[counts > 1].index:                       # keep every name unique
    for c in [c for c, n in names.items() if n == dupe]:
        diff = prof.loc[c] - prof.mean()
        d = diff.abs().idxmax()
        tag = ("above average on " if diff[d] > 0 else "below average on ") + SHORT_DOMAIN[d]
        names[c] = f"{names[c].replace(', no standout gap', '')}, {tag}"

for c, n in names.items():
    print(f"cluster {c}: {n}   [{(final_labels == c).sum()} districts]")

angles = np.linspace(0, 2 * np.pi, prof.shape[1], endpoint=False).tolist()
fig, axes = plt.subplots(2, int(np.ceil(K / 2)), figsize=(4.2 * np.ceil(K / 2), 9),
                         subplot_kw={"projection": "polar"})
for ax, c in zip(axes.ravel(), prof.index):
    vals = list(prof.loc[c]) + [prof.loc[c].iloc[0]]
    ax.plot(angles + [angles[0]], vals, lw=2)
    ax.fill(angles + [angles[0]], vals, alpha=0.25)
    ax.plot(angles + [angles[0]], [0] * (len(angles) + 1), color="#888", ls="--", lw=1)
    ax.set_xticks(angles)
    ax.set_xticklabels([DOMAIN_LABELS[d].split(" (")[0] for d in prof.columns], fontsize=6)
    ax.set_ylim(-1.5, 1.5); ax.set_yticks([-1, 0, 1]); ax.set_yticklabels([], fontsize=6)
    ax.set_title(f"{c}: {names[int(c)][:38]}\n({(final_labels == c).sum()} districts)", fontsize=8)
for ax in axes.ravel()[K:]:
    ax.axis("off")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "13_radars.png", dpi=120); plt.show()

## Cell 28 - How safe is each district's label?

Every profile has a centre, and a district takes the label of the nearest one. So
for any district you can ask how much further away the **second**-nearest centre
is:

* runner-up twice as far -> the district is deep inside its profile, nothing will
  move it;
* runner-up the same distance -> the district sits on a boundary and got its label
  by a hair.

This one number explains the unseen-state results in cell 30 before you even run
them, and it is the honest answer to "is this district really profile 4?".

In [ ]:
centroids_all = pd.DataFrame(final_scores).groupby(final_labels).mean().to_numpy()
dist_to_centres = np.linalg.norm(final_scores[:, None, :] - centroids_all[None, :, :], axis=2)
sorted_d = np.sort(dist_to_centres, axis=1)
margin = sorted_d[:, 1] / sorted_d[:, 0]        # 1.0 = exactly on a boundary

ids_margin = ids.assign(cluster=final_labels, margin=margin)
print(f"national median margin: {np.median(margin):.2f}x")
print(f"districts on a boundary (margin < 1.15): {(margin < 1.15).sum()} of {len(margin)}")

print("\nby state (the three used in the unseen-state test):")
for st in ["Bihar", "Karnataka", "Assam"]:
    m = ids_margin.loc[ids_margin.state.str.contains(st, case=False), "margin"]
    if len(m):
        print(f"  {st:10s} median {m.median():.2f}x | borderline {(m < 1.15).sum()}/{len(m)}")

print("\nmost borderline districts:")
display(ids_margin.nsmallest(5, "margin")[["district", "state", "cluster", "margin"]].round(2))
print("most solidly placed:")
display(ids_margin.nlargest(3, "margin")[["district", "state", "cluster", "margin"]].round(2))

plt.figure(figsize=(9, 4.5))
plt.hist(margin, bins=40, color="#2c7fb8")
plt.axvline(1.15, ls="--", color="#c7254e", label="boundary cases (< 1.15x)")
plt.xlabel("how much further the 2nd-nearest profile is")
plt.ylabel("districts"); plt.legend()
plt.title("Label confidence across all districts")
plt.tight_layout(); plt.savefig(ROOT / "figures" / "14_label_margin.png", dpi=120); plt.show()

## Cell 29 - Assigning a district the model has never seen

`assign_cluster` pushes any set of values through the same steps - fill gaps, clip, standardise, PCA, predict - refitting nothing. This is the function the Streamlit app calls.

In [ ]:
final_model = fit_kmeans(final_scores, K)
n_components = final_scores.shape[1] if final_space != "raw" else None
centroids = pd.DataFrame(final_scores).groupby(final_labels).mean().to_numpy()

def assign_cluster(values: dict):
    """values: {indicator_short_name: number} - any subset. Missing ones get imputed."""
    row = pd.Series({c: np.nan for c in X_raw.columns}, dtype=float)
    for k_, v in values.items():
        if k_ in row.index and v is not None and not pd.isna(v):
            row[k_] = float(v)
    imputed = [c for c in X.columns if pd.isna(row[c])]

    filled = impute_pipe.named_steps["impute"].transform(
        impute_pipe.named_steps["scale"].transform(row.to_frame().T))
    filled = impute_pipe.named_steps["scale"].inverse_transform(filled)
    one = pd.DataFrame(filled, columns=X_raw.columns)[X.columns].clip(lower=lo[X.columns],
                                                                     upper=hi[X.columns], axis=1)
    point = scaler.transform(one)
    if n_components:
        point = pca.transform(point)[:, :n_components]

    cluster = int(final_model.predict(point)[0])
    dists = np.linalg.norm(final_scores - point, axis=1)
    nearest = ids.assign(cluster=final_labels, distance=dists).nsmallest(5, "distance")
    return dict(cluster=cluster, cluster_name=names[cluster],
                distances={i: float(d) for i, d in enumerate(np.linalg.norm(centroids - point, axis=1))},
                nearest_districts=nearest, imputed_fields=imputed, point=point[0])

# sanity check: feed a real district back in and see if it lands where it should
check = assign_cluster(X.iloc[0].to_dict())
print(f"{ids.iloc[0]['district']} ({ids.iloc[0]['state']}): "
      f"assigned {check['cluster']}, actual {final_labels[0]}")
assert check["cluster"] == final_labels[0]

# a partially filled, hand-entered district
partial = {"women_10yr_schooling": 25, "improved_sanitation": 45, "clean_cooking_fuel": 20,
           "institutional_births": 60, "fully_vaccinated_card_or_recall": 55}
res = assign_cluster(partial)
print(f"\nhand-entered district -> cluster {res['cluster']}: {res['cluster_name']}")
print(f"({len(res['imputed_fields'])} of {len(X.columns)} indicators were imputed)")
display(res["nearest_districts"])

## Cell 29 - The unseen-state test

The closest thing an unsupervised project has to a test set: rebuild the scaler, the
PCA rotation **and** the clustering without one whole state, then assign that state's
districts with the reduced pipeline and see whether they land in the same profiles.

Expect uneven results, and read them with cell 28 open. A state whose districts sit
deep inside their profiles reproduces perfectly; a state whose districts sit on a
boundary flips as a block, because they all move to the same side together. That is
not a broken model - it is the continuum showing through, and it tells you which
districts' labels to trust.

In [ ]:
def unseen_state_test(state):
    held = (ids["state"] == state).to_numpy()
    if held.sum() < 2:
        return None
    sc = StandardScaler().fit(X[~held])
    Ztr = sc.transform(X[~held])
    if n_components:
        pc = PCA(random_state=SEED).fit(Ztr)
        Ptr, Pte = pc.transform(Ztr)[:, :n_components], pc.transform(sc.transform(X[held]))[:, :n_components]
    else:
        Ptr, Pte = Ztr, sc.transform(X[held])
    # refit the SAME algorithm the final model uses - refitting K-means while the
    # final model is a GMM would compare two different methods
    algo = SPECS[final_name]["algo"]
    if algo == "gmm":
        model = GaussianMixture(n_components=K, covariance_type="full" if n_components else "diag",
                                n_init=10, random_state=SEED).fit(Ptr)
        assigned = model.predict(Pte)
    elif algo == "ward":
        train_labels = AgglomerativeClustering(n_clusters=K, linkage="ward").fit(Ptr).labels_
        cents = np.vstack([Ptr[train_labels == c].mean(axis=0) for c in np.unique(train_labels)])
        assigned = np.argmin(((Pte[:, None, :] - cents[None, :, :]) ** 2).sum(axis=2), axis=1)
    else:
        assigned = KMeans(n_clusters=K, n_init=50, random_state=SEED).fit(Ptr).predict(Pte)
    truth = final_labels[held]
    pair_agreement = np.mean([(truth[i] == truth[j]) == (assigned[i] == assigned[j])
                              for i in range(len(truth)) for j in range(i + 1, len(truth))])
    return dict(state=state, n_districts=int(held.sum()),
                ari_vs_full_model=adjusted_rand_score(truth, assigned),
                pair_agreement=float(pair_agreement))

unseen = pd.DataFrame([r for r in (unseen_state_test(s) for s in ["Karnataka", "Bihar", "Assam"]) if r])
unseen = unseen.merge(
    ids_margin.groupby(ids_margin.state.str.replace("&", "").str.strip())["margin"]
              .median().rename("median_label_margin"),
    left_on="state", right_index=True, how="left")
display(unseen.round(3))
print("\nRead the two columns together: the states that reproduce badly are the ones")
print("whose districts sit closest to a boundary (margin near 1.0).")

## Cell 30 - Save everything

Writes the tables, the fitted models and a one-page summary into `nfhs_output/`. In Colab, download that folder from the file browser on the left.

In [ ]:
out = ids.copy()
out["cluster"] = final_labels
out["cluster_name"] = [names[int(c)] for c in final_labels]
out["composite_score"] = composite.to_numpy()
out["composite_quantile_group"] = groupings["composite_index"]
for d in district_domains.columns:
    out[f"domain_{d}"] = district_domains[d].to_numpy()

out.to_csv(ROOT / "processed" / "district_clusters.csv", index=False)
X.to_csv(ROOT / "processed" / "features.csv", index=False)
Y.to_csv(ROOT / "processed" / "outcomes.csv", index=False)
comparison.to_csv(ROOT / "processed" / "comparison.csv", index=False)
prof.to_csv(ROOT / "processed" / "cluster_domain_profiles.csv")
pairs.head(200).to_csv(ROOT / "processed" / "same_score_different_problems.csv", index=False)
joblib.dump({"scaler": scaler, "pca": pca, "model": final_model, "imputer": impute_pipe,
             "columns": list(X.columns), "n_components": n_components, "k": K,
             "cluster_names": names}, ROOT / "models" / "pipeline.joblib")

summary = f"""# Results

Districts: {len(out)} | features: {X.shape[1]} | held-out outcomes: {Y.shape[1]}
Final model: {final_name}, k = {K}

Held-out outcome variance explained (mean eta-squared)
  this clustering : {show.loc[final_name, 'mean_outcome_eta2']:.3f}
  composite index : {show.loc['composite_index', 'mean_outcome_eta2']:.3f}   (same number of groups)
  state grouping  : {show.loc['state', 'mean_outcome_eta2']:.3f}   ({int(show.loc['state', 'n_groups'])} groups, not like-for-like)

Profiles:
""" + "\n".join(f"  {c}: {n}  [{(final_labels == c).sum()} districts]" for c, n in names.items())
(ROOT / "SUMMARY.md").write_text(summary, encoding="utf-8")
print(summary)
print("\nsaved everything under:", ROOT.resolve())

---
## What to take away

1. **Clustering beats the single-score ranking** at explaining outcomes neither of
   them was allowed to see - by several times over, at the same number of groups.
2. **State grouping scores higher**, with ~34 groups against 8. Reported openly: more
   groups always score better on eta-squared, and "which state" still does not tell a
   health officer what a district needs.
3. **The districts are a continuum, not k natural types.** The silhouette around 0.10,
   DBSCAN's single blob, and the label margins in cell 28 all say the same thing. The
   profiles are practical labels cut across a smooth spread.
4. **So report the label with its margin.** Districts deep inside a profile keep their
   label under any rebuild (Bihar); districts on a boundary do not (Assam, Karnataka).
   That distinction is the most useful thing this analysis produces.

**Credit:** data from the [NFHS mirror by SaiSiddhardhaKalla](https://github.com/SaiSiddhardhaKalla/NFHS)
and the [factsheet mirror by jvargh7](https://github.com/jvargh7/nfhs5_factsheets), both parsed from
[IIPS NFHS factsheets](http://rchiips.org/nfhs/). Earlier PCA + K-means work on this data:
[kalyaninagaraj/NFHS5](https://github.com/kalyaninagaraj/NFHS5).